# Garmin — Setup & Initial Load

Self-contained notebook that:

1. Creates the Unity Catalog schema and bronze table (idempotent)
2. Reads the `garmin_oauth_tokens` secret from the Databricks secret scope
3. Authenticates to Garmin Connect using those tokens (no email/password)
4. Pulls one day of health data (yesterday by default)
5. Writes raw JSON payloads to the `wearables_zerobus` bronze table
6. Refreshes tokens back into the secret scope
7. Runs verification queries

**Prerequisite.** The `garmin_oauth_tokens` secret must exist in the scope before
running this notebook. Populate it from your local machine with:

```bash
./garmin/scripts/upload_garmin_tokens.sh --profile <your-cli-profile>
```

This notebook never prompts for Garmin email, password, or MFA. Credentials
are typed once into your local terminal by the upload script and are never
written to disk, passed as a job parameter, or committed to git.

In [ ]:
%pip install garth>=0.5 garminconnect>=0.2.0

In [ ]:
dbutils.library.restartPython()

## Configuration

These widgets carry non-credential configuration only. When run as a job,
the values come from job parameters; when run interactively, adjust them
in the widget bar above.

In [ ]:
DEFAULT_SCHEMA = "wearables"
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_DEVICE_ID = "garmin_device"

try:
    dbutils.widgets.text("catalog", "", "1. Catalog")
    dbutils.widgets.text("schema", DEFAULT_SCHEMA, "2. Schema")
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "3. Secret scope")
    dbutils.widgets.text("target_date", "", "4. Target date (YYYY-MM-DD, blank=yesterday)")
    dbutils.widgets.text("device_id", DEFAULT_DEVICE_ID, "5. Device ID")

    catalog = dbutils.widgets.get("catalog").strip()
    schema = dbutils.widgets.get("schema").strip() or DEFAULT_SCHEMA
    secret_scope = dbutils.widgets.get("secret_scope").strip() or DEFAULT_SECRET_SCOPE
    target_date_str = dbutils.widgets.get("target_date").strip()
    device_id = dbutils.widgets.get("device_id").strip() or DEFAULT_DEVICE_ID
except Exception:
    catalog = ""
    schema = DEFAULT_SCHEMA
    secret_scope = DEFAULT_SECRET_SCOPE
    target_date_str = ""
    device_id = DEFAULT_DEVICE_ID

if not catalog:
    raise ValueError(
        "The `catalog` parameter is required. Set it in the widget or pass it as a job parameter."
    )

spark.sql(f"USE CATALOG {catalog}")

bronze_table = f"{catalog}.{schema}.wearables_zerobus"

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Bronze table: {bronze_table}")
print(f"Secret scope: {secret_scope}")
print(f"Device ID:    {device_id}")

## Step 1 — Ensure schema and bronze table exist

Idempotent. Matches the DDL in `zeroBus/dbxW_zerobus_infra`.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema {catalog}.{schema} ready")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table}
    (
      record_id STRING NOT NULL COMMENT 'Server-generated GUID for each ingested record',
      ingested_at TIMESTAMP COMMENT 'Server-side ingestion timestamp',
      body VARIANT COMMENT 'Raw JSON payload stored as VARIANT',
      headers VARIANT COMMENT 'HTTP request headers as JSON',
      record_type STRING COMMENT 'Extracted from X-Record-Type header',
      CONSTRAINT wearables_zerobus_pk PRIMARY KEY (record_id)
    )
    USING DELTA
    COMMENT 'Bronze table for wearable health data'
    TBLPROPERTIES (
      'delta.enableChangeDataFeed' = 'true',
      'quality' = 'bronze'
    )
""")
print(f"Table {bronze_table} ready")

## Step 2 — Load Garmin OAuth tokens from the secret scope

Reads the `garmin_oauth_tokens` secret (a JSON object containing `oauth1`
and `oauth2` token payloads produced by `upload_garmin_tokens.sh`) and
resumes a `garth` session from them. No email, password, or MFA input
happens here.

In [ ]:
import json
import tempfile
from pathlib import Path

from garminconnect import Garmin

try:
    tokens_raw = dbutils.secrets.get(scope=secret_scope, key="garmin_oauth_tokens")
except Exception as exc:
    raise RuntimeError(
        f"Secret '{secret_scope}/garmin_oauth_tokens' not found. "
        "Run garmin/scripts/upload_garmin_tokens.sh from your local machine "
        "to provision it before running this notebook."
    ) from exc

try:
    tokens = json.loads(tokens_raw)
    oauth1 = tokens["oauth1"]
    oauth2 = tokens["oauth2"]
except Exception as exc:
    raise RuntimeError(
        "garmin_oauth_tokens is not in the expected {'oauth1': ..., 'oauth2': ...} "
        "format. Re-run garmin/scripts/upload_garmin_tokens.sh."
    ) from exc

session_dir = Path(tempfile.mkdtemp(prefix="garth_"))
(session_dir / "oauth1_token.json").write_text(json.dumps(oauth1))
(session_dir / "oauth2_token.json").write_text(json.dumps(oauth2))

client = Garmin()
client.login(str(session_dir))
print("Garmin Connect: authenticated via stored OAuth tokens")

## Step 3 — Determine target date

In [ ]:
from datetime import date, timedelta

if target_date_str:
    target_date = date.fromisoformat(target_date_str)
else:
    target_date = date.today() - timedelta(days=1)

print(f"Target date: {target_date}")

## Step 4 — Extract data from Garmin Connect

Pulls ten API categories for the target date. Each call returns raw JSON
that we'll persist verbatim into the bronze table.

In [ ]:
ds = target_date.isoformat()

EXTRACTORS = [
    ("daily_stats", "get_stats"),
    ("heart_rates", "get_heart_rates"),
    ("sleep", "get_sleep_data"),
    ("stress", "get_stress_data"),
    ("hrv", "get_hrv_data"),
    ("spo2", "get_spo2_data"),
    ("body_battery", "get_body_battery"),
    ("steps", "get_steps_data"),
    ("respiration", "get_respiration_data"),
]

raw_by_type = {}

for record_type, method_name in EXTRACTORS:
    try:
        raw_by_type[record_type] = getattr(client, method_name)(ds)
        print(f"  {record_type}: OK")
    except Exception as exc:
        print(f"  {record_type}: FAILED ({exc})")

try:
    raw_by_type["activities"] = client.get_activities_fordate(ds)
    print("  activities: OK")
except Exception as exc:
    print(f"  activities: FAILED ({exc})")

print(f"\nExtracted {len(raw_by_type)} categories for {ds}")

## Step 5 — Build bronze rows and write to the Delta table

In [ ]:
import uuid
from datetime import datetime, timezone


def to_bronze_row(raw_data, record_type, dev_id, target_ds):
    now = datetime.now(timezone.utc).isoformat()
    body = {
        "source": "garmin_connect",
        "device_id": dev_id,
        "date": target_ds,
        "data": raw_data,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Platform": "garmin_connect",
        "X-Record-Type": record_type,
        "X-Device-Id": dev_id,
        "X-Upload-Timestamp": now,
    }
    return {
        "record_id": str(uuid.uuid4()),
        "ingested_at": now,
        "body_json": json.dumps(body, default=str),
        "headers_json": json.dumps(headers),
        "record_type": record_type,
    }


bronze_rows = []
for record_type, raw_data in raw_by_type.items():
    if raw_data is None:
        continue
    bronze_rows.append(to_bronze_row(raw_data, record_type, device_id, ds))

print(f"Built {len(bronze_rows)} bronze rows:")
for row in bronze_rows:
    body_size = len(row["body_json"])
    print(f"  {row['record_type']:20s}  body={body_size:>6,} bytes")

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StringType, StructField, StructType

staging_schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("body_json", StringType(), False),
    StructField("headers_json", StringType(), False),
    StructField("record_type", StringType(), True),
])

rows = [Row(**r) for r in bronze_rows]
staging_df = spark.createDataFrame(rows, schema=staging_schema)
staging_df.createOrReplaceTempView("garmin_staging")

spark.sql(f"""
    INSERT INTO {bronze_table} (record_id, ingested_at, body, headers, record_type)
    SELECT
        record_id,
        CAST(ingested_at AS TIMESTAMP),
        parse_json(body_json),
        parse_json(headers_json),
        record_type
    FROM garmin_staging
""")

print(f"Wrote {len(bronze_rows)} rows to {bronze_table}")

## Step 6 — Persist refreshed OAuth tokens

`garth` silently refreshes the OAuth2 token during API calls. Push the
latest version back to the secret scope so the next run starts from a
fresh token.

In [ ]:
from databricks.sdk import WorkspaceClient

oauth1_refreshed = json.loads((session_dir / "oauth1_token.json").read_text())
oauth2_refreshed = json.loads((session_dir / "oauth2_token.json").read_text())
refreshed_payload = json.dumps({"oauth1": oauth1_refreshed, "oauth2": oauth2_refreshed})

WorkspaceClient().secrets.put_secret(
    scope=secret_scope,
    key="garmin_oauth_tokens",
    string_value=refreshed_payload,
)
print("Refreshed garmin_oauth_tokens stored in secret scope")

## Step 7 — Verify

Query the bronze table using VARIANT path expressions.

In [ ]:
print(f"=== {bronze_table} ===\n")
display(spark.sql(f"SELECT COUNT(*) AS total_rows FROM {bronze_table}"))

In [ ]:
print("--- Record types ---")
display(spark.sql(f"""
    SELECT
        record_type,
        COUNT(*) AS cnt,
        MIN(ingested_at) AS earliest,
        MAX(ingested_at) AS latest
    FROM {bronze_table}
    GROUP BY record_type
    ORDER BY cnt DESC
"""))

In [ ]:
print("--- Daily stats ---")
display(spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        body:data:restingHeartRate::INT AS resting_hr,
        body:data:totalSteps::INT AS steps,
        body:data:totalKilocalories::INT AS total_calories,
        body:data:activeKilocalories::INT AS active_calories,
        body:data:floorsAscended::INT AS floors,
        body:data:averageStressLevel::INT AS avg_stress,
        body:data:vO2MaxValue::DOUBLE AS vo2_max
    FROM {bronze_table}
    WHERE record_type = 'daily_stats'
    ORDER BY day DESC
"""))

In [ ]:
print("--- Sleep ---")
display(spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        ROUND(body:data:dailySleepDTO:sleepTimeSeconds::INT / 3600.0, 1) AS sleep_hours,
        ROUND(body:data:dailySleepDTO:deepSleepSeconds::INT / 60.0, 0) AS deep_min,
        ROUND(body:data:dailySleepDTO:lightSleepSeconds::INT / 60.0, 0) AS light_min,
        ROUND(body:data:dailySleepDTO:remSleepSeconds::INT / 60.0, 0) AS rem_min,
        body:data:dailySleepDTO:sleepScores:overall:value::INT AS sleep_score
    FROM {bronze_table}
    WHERE record_type = 'sleep'
    ORDER BY day DESC
"""))

In [ ]:
print("--- HRV ---")
display(spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        body:data:hrvSummary:weeklyAvg::DOUBLE AS hrv_weekly_avg_ms,
        body:data:hrvSummary:lastNight5MinHigh::DOUBLE AS hrv_last_night_ms,
        body:data:hrvSummary:status::STRING AS hrv_status
    FROM {bronze_table}
    WHERE record_type = 'hrv'
    ORDER BY day DESC
"""))

In [ ]:
print("--- All sources in bronze ---")
display(spark.sql(f"""
    SELECT
        headers:`X-Platform`::STRING AS platform,
        record_type,
        COUNT(*) AS cnt
    FROM {bronze_table}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

print("\nDone.")